<a href="https://colab.research.google.com/github/thenushans-magicktech/qwen2.5/blob/gemma/finetune/Gemma_finetune_on_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tune Gemma 4 E4B for the MagickHome kitchen designer (LoRA, free Colab T4)

Teaches Gemma the MagickHome playbook, rules and good designs **in its weights**, so the app can send the
**short prompt** (room + cabinet menu + customer needs, ~1,700 tokens) instead of the full knowledge pack (~6,600).

**Steps**
1. **Runtime -> Change runtime type -> T4 GPU -> Save.**
2. Check the settings cell below (repo / branch of the dataset, epochs). **Runtime -> Run all.**
   Training takes roughly 1-2 h for 300-500 examples. The model is saved to your Google Drive (allow access
   when asked), so the next session can skip training: set `SKIP_TRAINING = True`.
3. The last cell prints `python colab/connect_colab.py <url> <password> --finetuned` - run it on the laptop.

**New training data later:** add samples / generate more on the laptop, run `python finetune/merge.py`,
push the branch (or upload `train.jsonl` / `val.jsonl` here), then Run all again with `SKIP_TRAINING = False`.

## 0. Settings

In [ ]:
# Where the dataset comes from: the public GitHub repo (finetune/data/train.jsonl, val.jsonl) ...
REPO = "https://github.com/thenushans-magicktech/qwen2.5.git"
BRANCH = "gemma"
UPLOAD_INSTEAD = False      # ... or True: upload train.jsonl / val.jsonl from your laptop when asked

# Which base model to fine-tune (one setting). The Ollama name must start with "magick-gemma", "magick-lfm3" or
# "magick-qwen" so the app sends it the SHORT prompt (FINETUNED_MODEL_TAGS in config.py).
MODEL_CHOICE = "lfm3-1.7b"   # gemma4-e4b | gemma4-e2b | qwen3.5-2b | lfm3-1.7b
PRESETS = {
    "gemma4-e4b": ("unsloth/gemma-4-E4B-it", "magick-gemma"),       # Apache-2.0; quality reference (~1 h 50 m on a T4)
    "gemma4-e2b": ("unsloth/gemma-4-E2B-it", "magick-gemma-e2b"),   # Apache-2.0; ~2x faster on a laptop
    "qwen3.5-2b": ("unsloth/Qwen3.5-2B", "magick-qwen"),            # Apache-2.0; small and fast
    "lfm3-1.7b":  ("LiquidAI/LFM3-1.7B", "magick-lfm3"),            # LFM Open License (free < US$10M revenue); fastest
}
MAX_SEQ = 4096        # short prompt ~1,700 tokens + answer ~1,200
EPOCHS = 1            # 1 for the large rules-only set; 2-3 once it is mostly teacher / designer data
LORA_R = 16
LEARNING_RATE = 2e-4

SKIP_TRAINING = False   # True: reuse the model saved to Google Drive by an earlier session
SAVE_TO_DRIVE = False   # True: also copy the model to Google Drive (asks for Drive access)
DRIVE_DIR = "/content/drive/MyDrive/magick-gemma"

## 1. Check the GPU

In [ ]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("No GPU in this runtime. Runtime -> Change runtime type -> T4 GPU, then Runtime -> Run all.")
print("GPU:", gpu.stdout.strip())

## 2. Install Unsloth (fine-tuning) and get the dataset

In [ ]:
%%capture
!pip install -q unsloth

In [ ]:
import os, shutil, subprocess, json
os.makedirs("data", exist_ok=True)
if SAVE_TO_DRIVE or SKIP_TRAINING:
    from google.colab import drive
    drive.mount("/content/drive")
if not SKIP_TRAINING:
    if UPLOAD_INSTEAD:
        from google.colab import files
        print("Choose train.jsonl and val.jsonl")
        for name, data in files.upload().items():
            open(os.path.join("data", name), "wb").write(data)
    else:
        shutil.rmtree("repo", ignore_errors=True)
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO, "repo"], check=True)
        for name in ("train.jsonl", "val.jsonl"):
            shutil.copy(f"repo/finetune/data/{name}", f"data/{name}")
    for name in ("train.jsonl", "val.jsonl"):
        n = sum(1 for line in open(f"data/{name}") if line.strip())
        print(f"{name}: {n} examples")

## 3. Load Gemma 4 E4B in 4-bit and add LoRA adapters

In [ ]:
BASE_MODEL, OLLAMA_NAME = PRESETS[MODEL_CHOICE]
IS_GEMMA = "gemma" in BASE_MODEL.lower()
USE_UNSLOTH = True
print("Base model:", BASE_MODEL, "| Ollama name:", OLLAMA_NAME)
LOAD_FROM = BASE_MODEL
if "lfm3" in BASE_MODEL.lower():
    # LFM3 + newer transformers (5.17): its per-layer RoPE dict has no entry for 'conv' layers (KeyError 'conv'),
    # and transformers adds a stray top-level rope_theta that its own validator rejects. Patch both steps.
    import transformers.modeling_rope_utils as _mru
    _M = _mru.RotaryEmbeddingConfigMixin
    if not getattr(_M, "_magick_lfm3_patch", False):
        _std, _val = _M.standardize_rope_params, _M.validate_rope
        def _per_layer(self):
            rp = getattr(self, "rope_parameters", None)
            return rp if isinstance(rp, dict) and any(isinstance(v, dict) for v in rp.values()) else None
        def _standardize(self):
            rp = _per_layer(self)
            if rp is not None:
                for t in set(getattr(self, "layer_types", None) or []):
                    rp.setdefault(t, None)          # 'conv' layers use no RoPE
            return _std(self)
        def _validate(self):
            rp = _per_layer(self)
            if rp is not None:
                self.rope_parameters = {k: v for k, v in rp.items() if isinstance(v, dict) or v is None}
            return _val(self)
        _M.standardize_rope_params, _M.validate_rope, _M._magick_lfm3_patch = _standardize, _validate, True
        print("LFM3: transformers RoPE config patch applied")
if not SKIP_TRAINING:
    try:
        from unsloth import FastModel
        model, tokenizer = FastModel.from_pretrained(LOAD_FROM, max_seq_length=MAX_SEQ, load_in_4bit=True,
                                                     full_finetuning=False, trust_remote_code=True)
        extra = dict(finetune_vision_layers=False, finetune_language_layers=True,
                     finetune_attention_modules=True, finetune_mlp_modules=True) if IS_GEMMA else {}
        model = FastModel.get_peft_model(model, r=LORA_R, lora_alpha=LORA_R, lora_dropout=0, bias="none",
                                         random_state=3407, **extra)
    except Exception as exc:  # e.g. a model architecture Unsloth does not support yet
        print(f"Unsloth could not prepare {BASE_MODEL}: {exc}\n-> plain Hugging Face + PEFT (QLoRA)")
        USE_UNSLOTH = False
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
        tokenizer = AutoTokenizer.from_pretrained(LOAD_FROM, trust_remote_code=True)
        model = AutoModelForCausalLM.from_pretrained(
            LOAD_FROM, trust_remote_code=True, torch_dtype=torch.float16, device_map="auto",
            quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                   bnb_4bit_compute_dtype=torch.float16))
        model = prepare_model_for_kbit_training(model)
        model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_R, lora_dropout=0.0, bias="none",
                                                 target_modules="all-linear", task_type="CAUSAL_LM"))
    if hasattr(model, "print_trainable_parameters"):
        model.print_trainable_parameters()

## 4. Format the examples with Gemma's chat template (train only on the answers)

In [ ]:
if not SKIP_TRAINING:
    from datasets import load_dataset
    tok = getattr(tokenizer, "tokenizer", tokenizer)   # multimodal processor -> its text tokenizer
    ds = load_dataset("json", data_files={"train": "data/train.jsonl", "validation": "data/val.jsonl"})

    def render(msgs, gen=False):
        # thinking OFF, exactly as the app sends prompts (LFM3 / Qwen 3.5 templates think by default)
        try:
            text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=gen, enable_thinking=False)
        except TypeError:
            text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=gen)
        return text.removeprefix(tok.bos_token or "")

    ds = ds.map(lambda ex: {"text": render(ex["messages"])})
    # The template's turn markers, read from a placeholder message, so only the model's answer is learned.
    marked = render([{"role": "user", "content": "@@USER@@"}], gen=True)
    INSTRUCTION_PART = marked[:marked.index("@@USER@@")]
    RESPONSE_PART = marked[marked.index("@@USER@@") + len("@@USER@@"):]
    print("instruction marker:", repr(INSTRUCTION_PART), "| response marker:", repr(RESPONSE_PART))
    lengths = [len(tok(t)["input_ids"]) for t in ds["train"]["text"][:50]]
    print(f"tokens per example (first 50): max {max(lengths)}, mean {sum(lengths) // len(lengths)}")

## 5. Train

In [ ]:
if not SKIP_TRAINING:
    from trl import SFTTrainer, SFTConfig
    from unsloth.chat_templates import train_on_responses_only
    trainer = SFTTrainer(
        model=model, tokenizer=tok,
        train_dataset=ds["train"], eval_dataset=ds["validation"] if len(ds["validation"]) else None,
        args=SFTConfig(
            dataset_text_field="text", max_length=MAX_SEQ,
            per_device_train_batch_size=1, gradient_accumulation_steps=8,
            num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE, warmup_steps=5,
            lr_scheduler_type="linear", optim="adamw_8bit", weight_decay=0.01,
            logging_steps=5, eval_strategy="epoch" if len(ds["validation"]) else "no",
            seed=3407, report_to="none", output_dir="outputs", fp16=not USE_UNSLOTH,
        ),
    )
    trainer = train_on_responses_only(trainer, instruction_part=INSTRUCTION_PART, response_part=RESPONSE_PART)
    stats = trainer.train()
    print(stats)

## 6. Quick check: does it answer valid JSON on held-out rooms?

In [ ]:
if not SKIP_TRAINING:
    import torch
    FastModel.for_inference(model) if USE_UNSLOTH else model.eval()
    tests = ds["validation"] if len(ds["validation"]) else ds["train"]
    good = 0
    for ex in list(tests)[:3]:
        ids = tok(render(ex["messages"][:-1], gen=True), return_tensors="pt").to("cuda")
        out = model.generate(**ids, max_new_tokens=1500, do_sample=False)
        answer = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        try:
            json.loads(answer[answer.index("{"): answer.rindex("}") + 1]); good += 1; ok = "valid JSON"
        except Exception:
            ok = "NOT valid JSON (Ollama's schema lock fixes the format at run time)"
        print(ok, "|", answer[:300].replace("\n", " "), "...")
    print(f"{good}/3 valid without the schema lock")

## 7. Export to GGUF (for Ollama) and keep a copy on Google Drive

In [ ]:
import glob
from google.colab import files
GGUF = None
if not SKIP_TRAINING:
    # 1. the trained adapter (small) - downloaded first, so the work is safe even if the session ends
    model.save_pretrained("lora_adapter"); tokenizer.save_pretrained("lora_adapter")
    ADAPTER_ZIP = shutil.make_archive(f"/content/{OLLAMA_NAME}-lora", "zip", "lora_adapter")
    print("Adapter:", ADAPTER_ZIP, f"({os.path.getsize(ADAPTER_ZIP) / 1e6:.0f} MB) - downloading")
    files.download(ADAPTER_ZIP)
    # 2. the merged GGUF for Ollama (Unsloth models; ~15-25 min)
    if USE_UNSLOTH:
        try:
            model.save_pretrained_gguf("gguf_out", tokenizer, quantization_method="q4_k_m")
        except Exception as exc:
            print("GGUF export failed:", exc)
    else:
        print("Plain-PEFT model: no GGUF here - convert the adapter on the laptop (finetune/README.md).")
    if SAVE_TO_DRIVE:
        os.makedirs(DRIVE_DIR, exist_ok=True)
        for f in glob.glob("gguf_out/*.gguf") + [ADAPTER_ZIP]:
            shutil.copy(f, DRIVE_DIR)
        print("Saved to", DRIVE_DIR)
src_dir = DRIVE_DIR if SKIP_TRAINING else "gguf_out"
found = sorted(glob.glob(f"{src_dir}/*.gguf"), key=os.path.getsize)
GGUF = found[-1] if found else None
print("Model file:", GGUF, f"({os.path.getsize(GGUF) / 1e9:.1f} GB)" if GGUF else "- none (use the adapter zip)")

## 8. Serve it with Ollama

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os, subprocess, time, urllib.request
env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_KEEP_ALIVE="2h")
ollama_proc = subprocess.Popen(["ollama", "serve"], env=env,
                               stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434", timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    print(open("ollama.log").read()[-3000:])
    raise SystemExit("Ollama did not start - its log is printed above")
print("Ollama is running")

In [ ]:
assert GGUF, "No GGUF to serve - use the downloaded adapter zip on the laptop (finetune/README.md)"
# Gemma: Ollama's built-in Gemma 4 renderer / parser; other models: the chat template inside the GGUF
lines = [f"FROM {GGUF}"] + (["RENDERER gemma4", "PARSER gemma4"] if IS_GEMMA else [])
lines += ["PARAMETER temperature 0", "PARAMETER num_ctx 8192"]
open("Modelfile.magick", "w").write("\n".join(lines) + "\n")
print(open("Modelfile.magick").read()[:800])
r = subprocess.run(["ollama", "create", OLLAMA_NAME, "-f", "Modelfile.magick"], capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])
assert r.returncode == 0, "ollama create failed - see above"
MODEL = OLLAMA_NAME

In [ ]:
import json, time, urllib.request
t = time.time()
body = json.dumps({"model": MODEL, "prompt": "Reply with the single word OK.",
                   "stream": False, "think": False, "options": {"num_predict": 5}}).encode()
req = urllib.request.Request("http://127.0.0.1:11434/api/generate", data=body,
                            headers={"Content-Type": "application/json"})
answer = json.load(urllib.request.urlopen(req, timeout=600))["response"].strip()
print(f"Model loaded and answered {answer!r} in {time.time() - t:.0f} s")
!ollama ps

## 9. Password proxy and public tunnel (same as the Gemma notebook)

In [ ]:
%%writefile ollama_auth_proxy.py
"""Password (token) protected reverse proxy in front of an Ollama server - standard library only.

Runs next to Ollama on the GPU machine (Google Colab, a rented GPU, an office server).
Only requests carrying ``Authorization: Bearer <OLLAMA_API_KEY>`` reach Ollama, so a public
tunnel URL alone is not enough to use the GPU. The kitchen generator sends that header
automatically when ``OLLAMA_API_KEY`` is set on the laptop (the ``ollama`` Python client reads it).

    OLLAMA_API_KEY=<secret> python ollama_auth_proxy.py <port>

No third-party packages: works on any Python 3.8+ (including Colab's) without pip installs.
"""
import hmac
import http.client
import json
import os
import sys
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

UPSTREAM_HOST = os.environ.get("OLLAMA_UPSTREAM_HOST", "127.0.0.1")
UPSTREAM_PORT = int(os.environ.get("OLLAMA_UPSTREAM_PORT", "11434"))
TOKEN = os.environ["OLLAMA_API_KEY"]
DROP = {"host", "authorization", "content-length", "connection", "accept-encoding", "transfer-encoding"}


class Proxy(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"

    def _reply(self, status: int, body: bytes, ctype: str = "application/json") -> None:
        self.send_response(status)
        self.send_header("Content-Type", ctype)
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        if self.command != "HEAD":
            self.wfile.write(body)

    def _forward(self) -> None:
        if not hmac.compare_digest(self.headers.get("Authorization", ""), f"Bearer {TOKEN}"):
            return self._reply(401, b'{"error":"unauthorized"}')
        length = int(self.headers.get("Content-Length") or 0)
        data = self.rfile.read(length) if length else None
        headers = {k: v for k, v in self.headers.items() if k.lower() not in DROP}
        try:
            conn = http.client.HTTPConnection(UPSTREAM_HOST, UPSTREAM_PORT, timeout=None)
            conn.request(self.command, self.path, body=data, headers=headers)
            resp = conn.getresponse()
            body = resp.read()
            ctype = resp.getheader("Content-Type", "application/json")
            conn.close()
        except OSError as exc:
            return self._reply(502, json.dumps({"error": f"ollama not reachable: {exc}"}).encode())
        self._reply(resp.status, body, ctype)

    do_GET = do_POST = do_DELETE = do_HEAD = _forward

    def log_message(self, fmt, *args):  # one short line per request
        sys.stderr.write("%s %s\n" % (self.address_string(), fmt % args))


if __name__ == "__main__":
    port = int(sys.argv[1]) if len(sys.argv) > 1 else 11435
    print(f"Ollama password proxy on 127.0.0.1:{port} -> {UPSTREAM_HOST}:{UPSTREAM_PORT}", flush=True)
    ThreadingHTTPServer(("127.0.0.1", port), Proxy).serve_forever()

In [ ]:
import os, secrets, socket, subprocess, sys, time, urllib.error, urllib.request
try:
    proxy_proc.kill()  # re-running this cell: stop the previous proxy first
except NameError:
    pass
with socket.socket() as s:  # a free port (Colab uses some ports, e.g. 8080, for itself)
    s.bind(("127.0.0.1", 0))
    PORT = s.getsockname()[1]
TOKEN = secrets.token_urlsafe(24)
proxy_proc = subprocess.Popen([sys.executable, "ollama_auth_proxy.py", str(PORT)],
                              env=dict(os.environ, OLLAMA_API_KEY=TOKEN),
                              stdout=open("proxy.log", "w"), stderr=subprocess.STDOUT)
alive = False
for _ in range(20):
    time.sleep(1)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/tags", timeout=3)
    except urllib.error.HTTPError as e:
        alive = e.code == 401  # answering, and refusing requests without the password
        break
    except Exception:
        if proxy_proc.poll() is not None:
            break
if not alive:
    print(open("proxy.log").read()[-3000:])
    raise SystemExit("Proxy did not start - its log is printed above")
print(f"Password proxy running on port {PORT}")

In [ ]:
import json, re, subprocess, time, urllib.request
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
try:
    tunnel_proc.kill()  # re-running this cell: stop the previous tunnel first
except NameError:
    pass
tunnel_proc = subprocess.Popen(["./cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
                               stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
URL = None
for _ in range(60):
    time.sleep(1)
    found = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("tunnel.log").read())
    if found:
        URL = found.group(0)
        break
if not URL:
    print(open("tunnel.log").read()[-3000:])
    raise SystemExit("Tunnel did not start - its log is printed above")

models = None
for _ in range(30):  # a brand-new tunnel URL can take a few seconds to resolve
    try:
        req = urllib.request.Request(URL + "/api/tags", headers={"Authorization": f"Bearer {TOKEN}"})
        models = [m["name"] for m in json.load(urllib.request.urlopen(req, timeout=15))["models"]]
        break
    except Exception:
        time.sleep(3)
print("Tunnel check:", "OK - models " + ", ".join(models) if models else "not answering yet (try the laptop step anyway)")
print()
print("On your laptop, in the project folder, run:")
print()
print(f"python colab/connect_colab.py {URL} {TOKEN} --finetuned")

## 10. Status

In [ ]:
for name, proc in (("ollama", ollama_proc), ("proxy", proxy_proc), ("tunnel", tunnel_proc)):
    print(f"{name:7s}", "running" if proc.poll() is None else f"STOPPED (exit {proc.returncode})")
!ollama ps
!tail -n 3 ollama.log

## When you are done
**Runtime -> Disconnect and delete runtime.** The trained model stays on Google Drive (`DRIVE_DIR`):
next time set `SKIP_TRAINING = True` and Run all - no training, ready in a few minutes.